# DATA Pre-Processing (Dataset + Dataloader)

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

class DiagnosticsDataset:
    def __init__(self, pdf_path):
        self.loader = PyPDFLoader(pdf_path)
        self.documents = self.loader.load()

    def __len__(self):
        return len(self.documents)

    def __getitem__(self, idx):
        return self.documents[idx].page_content


class DiagnosticsDataLoader:
    def __init__(self, dataset, chunk_size=800, overlap=100):
        self.dataset = dataset
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=overlap
        )

    def load(self):
        all_text = [self.dataset[i] for i in range(len(self.dataset))]
        return self.splitter.create_documents(all_text)


### check

In [14]:
ds = DiagnosticsDataset("ISO_14229.pdf")
d1 = DiagnosticsDataLoader(ds)
docs = d1.load()
print(len(docs))

1480


## Tokenization

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokens = tokenizer(
    "DiagnosticSessionControl service explanation",
    return_tensors="pt"
)

print(tokens["input_ids"].shape)

torch.Size([1, 7])


## Embeddings (Transformer ENCODER)

In [6]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(),override=True)

embeddings = OpenAIEmbeddings()

vector = embeddings.embed_query(
    "Explain UDS DiagnosticSessionControl"
)


## FAISS Vector DATABASE

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

def build_faiss_index(docs):
    embeddings = OpenAIEmbeddings()
    db = FAISS.from_documents(docs, embeddings)
    db.save_local("faiss_diagnostics")
    return db


In [8]:
db = build_faiss_index(docs)

## Retriever + RAG

In [9]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
db = FAISS.load_local("faiss_diagnostics", embeddings,allow_dangerous_deserialization=True)

retriever = db.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

pdf_rag = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)


In [10]:
import os
from dotenv import load_dotenv
from langchain_community.utilities import SerpAPIWrapper

load_dotenv(find_dotenv(),override=True)

search = SerpAPIWrapper(
    serpapi_api_key=os.getenv("SERPAPI_KEY")
)

## Research Agent(PDF + WEB + Reasoning)

In [11]:
def research_agent(question):
    pdf_context = pdf_rag.run(question)
    web_context = search.run(question)

    final_prompt = f"""
You are an automotive diagnostics research agent.

ISO 14229 Context:
{pdf_context}

Live Web Context:
{web_context}

Give a precise, standards-correct answer.
"""

    response = llm.invoke(final_prompt)
    return response.content

## Streamlit app

In [12]:
import streamlit as st

st.set_page_config("Diagnostics Research Agent")
st.title("🚗 ISO 14229 Diagnostics Research Agent")

query = st.text_input(
    "Ask diagnostics question (UDS, NRC, DTC, Services)"
)

if st.button("Research"):
    with st.spinner("Analyzing ISO 14229..."):
        answer = research_agent(query)
        st.markdown(answer)

2025-12-31 14:26:52.346 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 14:26:52.348 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 14:26:53.141 
  command:

    streamlit run c:\Users\Kaliammal\miniconda3\envs\tf\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-12-31 14:26:53.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 14:26:53.143 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 14:26:53.144 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-31 14:26:53.145 Thread 'MainThread': missing ScriptRunContext! This warning can be igno

In [16]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.utilities import SerpAPIWrapper

search = SerpAPIWrapper(
    serpapi_api_key=os.getenv("SERPAPI_KEY")
)

result = search.run("ISO 14229 DiagnosticSessionControl")
print(result)

ValueError: Got error from SerpAPI: Invalid API key. Your API key should be here: https://serpapi.com/manage-api-key